ARTI308 - Machine Learning

# Project Notebook 6: Random Forest
## Heart Disease Prediction  |  Model type: **ensemble**

**Goal:** train a Random Forest classifier to predict `HeartDisease` (0 / 1), evaluate it, and save its results so that notebook 08 can compare all four models.

Run notebooks 01, 02 and 03 first (notebook 03 creates `data/processed/heart_features.csv`).

## 1. Shared setup (identical in all model notebooks)

> **Do not change this section.** Every model notebook uses exactly the same data, the same features, the same train/test split (`random_state=42`) and the same preprocessing. This is what makes the comparison between models **fair**.

The input is `heart_features.csv` from notebook 3 (duplicates removed, hidden zeros converted to `NaN`, plus the new features `age_group` and `maxhr_pct`).
Imputation and scaling are done inside the pipeline, after the split, to avoid data leakage.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

In [ ]:
# 1) Load the data with the engineered features (created in notebook 03)
df = pd.read_csv("../data/processed/heart_features.csv")

# 2) Features and target
target_col = "HeartDisease"
X = df.drop(columns=[target_col])
y = df[target_col]

# 3) Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
numeric_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak", "FastingBS", "maxhr_pct"]
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope", "age_group"]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # fill NaN with the training median
    ("scaler", StandardScaler()),                    # z-score using training mean/std
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_pipe, numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. About the model: Random Forest
- An **ensemble** of many decision trees (here 300).
- Each tree is trained on a **random sample** of the rows and considers a **random subset** of the features at each split. This technique is called **bagging**.
- The final prediction is the **majority vote** of all trees, which reduces the overfitting of a single tree.
- Handles mixed features well and gives **feature importance**.
- Does **not need scaling** (the shared pipeline scales anyway; this does not hurt).

## 3. Build the pipeline and train

In [ ]:
MODEL_NAME = "Random Forest"
MODEL_TYPE = "ensemble"
FILE_KEY = "random_forest"
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
pipe.fit(X_train, y_train)
pipe

## 4. Evaluate on the test set

In [ ]:
y_pred = pipe.predict(X_test)

print("Accuracy: ", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:   ", round(recall_score(y_test, y_pred), 4))
print("F1-score: ", round(f1_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=["No disease (0)", "Disease (1)"]))

**Observation:**

*(Write what you notice after running the cell above.)*

### Confusion matrix
Rows = actual, columns = predicted.

- top-left: healthy, predicted healthy (correct)
- top-right: healthy, predicted sick (**false positive**)
- bottom-left: sick, predicted healthy (**false negative**, the dangerous mistake)
- bottom-right: sick, predicted sick (correct)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No disease", "Disease"], yticklabels=["No disease", "Disease"])
plt.title(f"Confusion Matrix - {MODEL_NAME}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 5. Cross-validation
One train/test split can be lucky or unlucky. With **5-fold stratified cross-validation** the training data is split 5 times and the scores are averaged.
The preprocessing is inside the pipeline, so the imputer and scaler are re-fit on every training fold (no leakage).

In [ ]:
cv_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
cv_rec = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall")

print(f"CV accuracy: {cv_acc.mean():.3f} (+/- {cv_acc.std():.3f})")
print(f"CV recall:   {cv_rec.mean():.3f} (+/- {cv_rec.std():.3f})")

**Observation:**

*(Write what you notice after running the cell above.)*

## 6. Feature importance
Because of one-hot encoding, every category becomes its own feature, so we first recover the feature names.

In [ ]:
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
importances = pipe.named_steps["model"].feature_importances_

fi = (pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False))

plt.figure(figsize=(8, 6))
sns.barplot(data=fi.head(15), x="importance", y="feature")
plt.title("Top 15 Feature Importances - " + MODEL_NAME)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

fi.head(15)

A high importance means the feature gave useful signal to the model; it does **not** prove a causal relationship.

**Observation:**

*(Write what you notice after running the cell above.)*

## 7. Save the results
The metrics are saved to `outputs/` so that notebook 08 can compare all models.

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

metrics = pd.DataFrame([{
    "model": MODEL_NAME,
    "type": MODEL_TYPE,
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "cv_accuracy": cv_acc.mean(),
    "cv_recall": cv_rec.mean(),
    "TN": tn, "FP": fp, "FN": fn, "TP": tp,
}])

os.makedirs("../outputs", exist_ok=True)
metrics.to_csv(f"../outputs/metrics_{FILE_KEY}.csv", index=False)
metrics.round(3)

---
### Tasks
1. Change `n_estimators` (for example 10, 100, 300) and compare accuracy and recall. Does adding more trees keep helping?
2. Compare the top features here with the Decision Tree notebook. Are they similar?